# C — 🏆 ML4NAT Competition

> Machine learning for Natural Sciences, SS2026, Prof. Pascal Friederich, pascal.friederich@kit.edu
>
> Competition window: **23 June 2026** – **13 July 2026**
>
> Submission deadline: **13 July 2026, 08:00**
>
> Tutor: jonas.teufel@kit.edu
>
> **Please ask questions in the forum/discussion board and only contact the Tutor directly when there are issues with the grading**

**Topic:** An *optional*, standalone competition: build a (machine learning) model that regresses **two** continuous material properties ($A$, $P$) from *colored diffractogram* images — concentric rings whose **position, thickness and color** jointly encode the properties — while coping with a strong distribution shift between clean training data and distorted test data.

Please add here your group members' names and student IDs.

You are encouraged to work in groups of a maximum of 3 people, however **each of you** has to submit a solution.

Names:

IDs:

## ⚠️ Important Information

This notebook is the **🏆 ML4NAT Competition** — a *standalone, entirely optional* exercise. Unlike the regular exercise sheets, it is not graded in the usual way: instead, you develop a model for the task described below and earn **bonus points** based on the absolute performance of your model on a held-out test set. These bonus points can push your overall exercise percentage **beyond 100%**, contributing accordingly to your final average. On top of the bonus points, the top submissions can win prizes (see C.1).

Participation is completely voluntary — but it's a great opportunity to apply everything you've learned so far on an open-ended, real-world-style problem.

---

**Installing dependencies.** The cell below installs the additional ``pytorch_lightning`` dependency, which is not part of the default environment.

In [ ]:
##### DO NOT CHANGE #####
%pip install lightning

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
import os
import json
import zipfile
import random
import requests
import tempfile
import importlib.util
from rich.pretty import pprint
from itertools import islice
from typing import Any, Dict, List, Set, Optional, Tuple, Literal

import PIL
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
import lightning.pytorch as pl
from PIL.Image import Image as Image_
from PIL import Image

PATH: str = os.getcwd()
plt.style.use('default')

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Will be used to download assets from the BwSync&Share Nextcloud.
import urllib3.util.connection

# Force IPv4 for outbound downloads. bwsyncandshare.kit.edu's AAAA record
# points to an IPv6 endpoint that is unreachable from many networks (including
# bwJupyter); Python's HTTP clients don't implement Happy Eyeballs and would
# otherwise wait the full TCP timeout (~120s) before falling back to v4.
urllib3.util.connection.HAS_IPV6 = False

def nextcloud_download(url: str, raw: bool = False) -> str:
    """
    Downloads the content of a file from a nextcloud server and returns 
    it eithers as a string or a bytes object if the ``raw`` flag is set.
    """
    
    content = b''
    with requests.get(
        url,
        allow_redirects=True,
        headers={
            'User-Agent': 'Mozilla/5.0',
        },
        stream=True,
        timeout=60,
    ) as response:
        for index, chunk in enumerate(response.iter_content(chunk_size=8192 * 1000)):
            if chunk:
                content += chunk
        
    print(f"> Downloaded {len(content)} bytes from {url}")
    if not raw:
        content = content.decode('utf-8')
    
    return content

# Can be used to import python modules dynamically from nextcloud.
def nextcloud_import(module_name: str, url: str) -> Any:
    """
    Downloads a python module from a nextcloud server and imports it.
    """
    
    content: str = nextcloud_download(url, raw=True)
    module_path: str = os.path.join(tempfile.gettempdir(), f'{module_name}.py')
    with open(module_path, 'wb') as file:
        file.write(content)
    
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    print('> Imported module:', module_name)
    
    return module



##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
utils = nextcloud_import(
    module_name='utils',
    url='https://bwsyncandshare.kit.edu/s/x22fQopd9YCj5t3/download'
)

##### DO NOT CHANGE #####

## C.1 How the Competition Works

Participation in this competition is completely **optional**. The points awarded are *bonus* points which may increase the final percentage of your exercise score beyond 100%—contributing to the final average accordingly.

The competition runs from **23 June 2026** until **13 July 2026**. The competition notebook has to be submitted to ILIAS by **13 July 2026, 08:00** at the latest.

In this competition you'll have to solve a regression task, possibly applying all of the machine learning knowledge you've acquired in the lecture so far, as well as your own creative problem-solving skills. Further down, you'll receive the detailed instructions along with a training dataset. Your task is to develop a (machine learning) method to solve the given problem. After submitting the notebook, your model will be used to obtain predictions for a *held-out* test set. By comparing the predictions of your model with the true target values, your model's accuracy is measured *separately on each of three held-out subsets*. These per-subset accuracies feed **two independent outcomes** — your *bonus points* and your *prize ranking* — both explained below.

**Two ways your model is scored, one set of measurements.** In both cases your model is evaluated on the **same three held-out subsets**, and the resulting accuracies ($R^2$) are simply turned into a result in two different ways:

- **🎯 Bonus points** — count toward your exercise grade. Each subset is checked against fixed thresholds and scored *independently*, for up to **70** points in total (see *Bonus-Point Grading* below).
- **🏆 Prize ranking** — decides the prizes. All submissions are ranked by a **single number: the mean $R^2$ averaged over all three subsets** (across both properties $A$ and $P$) (see *Prizes* below).

Both are computed from the exact same datasets and the exact same predictions — they differ only in *how* those $R^2$ scores are turned into a result.

### 🎁 Prizes

In total, there are **three** prizes available, awarded to the top submissions in the **Competition Ranking** (explained below). This year the prizes are **M5Stack microcontroller dev kits** — small, programmable gadgets you get to take home and build your own projects with:

- **🥇 1st Place — M5Stack Core2:** a pocket-sized, programmable ESP32 mini-computer with its own colour *touchscreen*, speaker and motion sensor — ready to turn into whatever gadget you can dream up.
- **🥈 2nd Place — M5StickS3:** a tiny stick-shaped ESP32 controller with a little display, microphone and a *built-in battery* — perfect for portable IoT and voice projects.
- **🥉 3rd Place — M5StickS3:** another M5StickS3, identical to the 2nd-place prize.

<table style="border: none; margin: 8px 0;">
<tr>
<td style="text-align: center; border: none; padding: 0 18px;">
<img src="https://shop.m5stack.com/cdn/shop/files/1_3129c02c-c533-4fbd-99f7-e8ae7695b5a6_1200x1200.webp" alt="M5Stack Core2" width="210"><br>
<strong>🥇 M5Stack Core2</strong>
</td>
<td style="text-align: center; border: none; padding: 0 18px;">
<img src="https://shop.m5stack.com/cdn/shop/files/1_f50b328d-f753-4bef-92ec-6f0e8a69e5c0_1200x1200.webp?v=1769049123" alt="M5StickS3" width="210"><br>
<strong>🥈🥉 M5StickS3</strong>
</td>
</tr>
</table>

*If* you win a prize, you'll be notified by an additional email—separate from the exercise's grading results! To receive your prize you'll *have to* attend the in-person exercise on **17 July 2026**.

<div style="border: 1px solid #C75949; border-radius: 3px; padding: 6px; background-color: #fbeae5; color: black;">
<strong>⚠️ NOTE — Working Together.</strong> For the competition, you are still welcome to collaborate with your fellow students to develop the general methodology of your solution. However, in the end, each of you will have to train and submit their own independent model. If two solutions yield the <em>exact same</em> predictions they will be <em>disqualified</em> from the competition!
</div>

### 📝 Bonus-Point Grading

Independently of the prizes, you can earn *bonus points* for the exercise. These points are added on top of your **final exercise average** at the end of the semester. Crucially, the competition does **not** count as one of your regular exercise notebooks: it neither increases the number of notebooks your average is taken over, nor can it be among the worst notebooks that get dropped. Instead, the points you collect here are added *directly into* your final average — so they can push your final score (and therefore your grade) up, potentially **beyond 100%**.

You collect points **independently** in each of the three evaluation scenarios. For each scenario there are **two thresholds** on that scenario's mean coefficient of determination $\bar R^2 = \tfrac{1}{2}(R^2_A + R^2_P)$ — a *lower* ceiling that is relatively easy to reach, and a *higher* ceiling that is hard. You do not need to do well on the easier scenarios to score on the harder ones; every scenario is graded on its own.

| Scenario | Lower ceiling | Higher ceiling |
|---|---|---|
| **Test** | $\bar R^2 \ge 0.80$ → **5 pts** | $\bar R^2 \ge 0.90$ → **10 pts** |
| **Basement (easy)** | $\bar R^2 \ge 0.85$ → **10 pts** | $\bar R^2 \ge 0.95$ → **15 pts** |
| **Basement (hard)** | $\bar R^2 \ge 0.75$ → **10 pts** | $\bar R^2 \ge 0.85$ → **20 pts** |

In total you can collect up to **70 points**, which corresponds to roughly **+7 percentage points** on your final exercise average (the exact amount depends on how many exercises there are in the semester). Even clearing only the three *lower* ceilings already gives you a noticeable boost.

### 🏆 Competition Ranking

Separately from the bonus points, all submissions are also entered into a **ranking** that decides the prizes. The ranking is based on a single overall accuracy score — the **mean $R^2$ across all three held-out subsets**, weighting both target properties equally:

$$\text{ranking score} \;=\; \tfrac{1}{3}\left(\bar R^2_{\text{test}} + \bar R^2_{\text{easy}} + \bar R^2_{\text{hard}}\right), \qquad \bar R^2_{\text{subset}} = \tfrac{1}{2}\left(R^2_A + R^2_P\right)$$

So every subset and both properties count equally. Submissions are sorted by this single number (higher is better), and the top three receive the prizes listed above. Unlike the bonus points, there are **no thresholds** here — even small differences in accuracy can change your rank.

## C.2 🎯 The Task

<div style="border: 1px solid #C75949; border-radius: 3px; padding: 6px; background-color: #fbeae5; color: black;">
<strong>⚠️ Disclaimer — this is a made-up task.</strong> The scenario described below is entirely fictional and was invented purely for this exercise. The "diffractograms", the "amplitude" $A$ and "phase" $P$ properties, and the story about the physicists and their basement archive are <em>not</em> real. They are only <em>loosely</em> inspired by real measurement techniques (such as powder X-ray diffraction) and do <em>not</em> represent any actual physical experiment, instrument or dataset.
</div>

### Motivation

You are collaborating with a group of experimental physicists who probe two coupled material properties: an **amplitude** $A \in [0, 10]$ and a **phase** $P \in [-3, 3]$. Their instrument cannot read either property out directly. Instead, it fires a short *photon packet* at a thin slice of the material and uses a camera to record how the photons scatter as they pass through. The resulting image — a *diffractogram* — shows a set of **colored concentric rings** on a dark background. The **location (radius)**, **thickness** and **color** of the individual rings together encode the two properties $A$ and $P$ in a complex, implicit way.

The physicists provide you with a large dataset of annotated diffractograms for a number of materials whose values of $A$ and $P$ have been carefully measured. Your task is to train a model to recover both values from the diffractogram image alone.

After you've successfully implemented your model, your collaborators would like to apply it to *future* diffractograms for which $A$ and $P$ aren't already known. In addition, they dug up a box of old diffractogram images from the basement of the institute, left behind by previous generations of researchers and recorded on *older measurement devices*. They would be particularly interested to use your model on these older records as well — a special challenge, since the older equipment introduced significantly more irregularities and artifacts into the images!

Through a significant amount of manual work, the physicists were able to determine the true values $A$ and $P$ for a small subset of **25** of these basement diffractograms which you'll be able to use to validate your model.

### Task Description

You receive a *training* dataset of **2000 annotated** diffractograms. Each element is a $512\times 512$ pixel RGB image saved in the PNG format, associated with the true property pair $(A, P)$. You may use this dataset as you wish to train your model / develop your method.

Your task is to develop a model which takes a diffractogram image as input and produces a prediction of **both** continuous properties $(A, P)$ as output. The properties are encoded jointly in the **position, thickness and color** of the rings — so discarding the color information (e.g. by converting the image to grayscale) may throw away part of the signal.

Specifically, your model will have to implement the ``DiffractogramInterface`` which is shown below.

Upon submission, your model will be evaluated on **three held-out subsets of 250 diffractograms each**, in order of increasing difficulty:

1. **Test set (250 images)** — Your model is first tested on 250 fresh   diffractograms recorded on the physicists' *current* measurement   device. These are clean, undistorted images of the same kind as your   training data — same instrument, same imaging quality.
2. **Basement (easy) (250 images)** — Next, your model is   evaluated on 250 diffractograms recovered from the basement archive.   These were recorded on older equipment that introduced **exactly   one** of 5 different types of distortion per image. The 5 distortion   types are equally represented (50 images per type). The distortion   types are **not named** for you — identifying them yourself by   studying the 25 annotated validation samples is part of the   challenge.
3. **Basement (hard) (250 images)** — Finally, your model faces the   toughest collection: 250 diffractograms from the deepest part of   the archive, each affected by **two** distinct distortions (drawn   from the same 5 types) stacked on top of each other in random   order.

### How Your Score Is Calculated

Your model is evaluated **separately** on each of the three subsets. For a given subset we compute $R^2_A$ and $R^2_P$ — the coefficients of determination for the two targets — and average them into that subset's mean score:

$$ \bar R^2_{\text{subset}} \;=\; \tfrac{1}{2}\left(R^2_A + R^2_P\right)_{\text{subset}} $$

Each subset's $\bar R^2$ is then compared independently against the two ceilings shown below to award bonus points.

The ceilings, and the points awarded for clearing each, are:

| Subset | Lower ceiling | Higher ceiling |
|---|---|---|
| **Test** (clean) | $\bar R^2 \ge 0.80$ → **5 pts** | $\bar R^2 \ge 0.90$ → **10 pts** |
| **Basement (easy)** | $\bar R^2 \ge 0.85$ → **10 pts** | $\bar R^2 \ge 0.95$ → **15 pts** |
| **Basement (hard)** | $\bar R^2 \ge 0.75$ → **10 pts** | $\bar R^2 \ge 0.85$ → **20 pts** |

For the **bonus points**, there is no single combined score — each subset is scored against its own ceilings independently, so strong performance on any one subset is rewarded on its own. The **prize ranking** (see C.1) works differently: it *does* use a single combined number — the **mean $R^2$ averaged over all three subsets** — to rank submissions for the prizes.

### 💡 Hints

✳️ **Distribution Shift.** It should be relatively easy to achieve very high accuracy on the *training* samples. The primary challenge of this competition is generalizing to the basement subsets, where the older equipment introduced significant distortions. Naïve machine-learning approaches will see severe performance drops on these samples compared to your training accuracy.

✳️ **Possible Solutions.** You are strongly encouraged to solve the task using some kind of neural network model, as it will get you good results the quickest. However, you are not *obligated* to use deep learning methods. You may also try traditional machine learning approaches or computer vision techniques. Ultimately, the best solution will likely consist of a clever combination of traditional pre-processing and machine learning.

✳️ **Validation Samples.** The 25 basement validation samples cover **all** of the 5 distortion types present in the **basement (easy)** subset (roughly 5 samples per type). The validation samples themselves carry **only** the $(A, P)$ labels — they are *not* annotated with their distortion type. Identifying the 5 types from the validation samples is part of the task.

✳️ **Available Libraries.** To implement your model and algorithms you may only use libraries which are available in the given environment by default and the additional ``pytorch_lightning`` package introduced at the beginning of this notebook. Most importantly, the available packages include the common machine learning frameworks ``torch``, ``tensorflow`` and ``jax`` along with basic libraries such as ``numpy``, ``pandas`` and ``scikit-learn``. In total, these available libraries will be more than sufficient to solve the given task.

✳️ **Remove Training Code.** During the grading of your notebook, each cell has a certain (relatively low) execution timeout. If the execution of the cell exceeds this timelimit it will be forcefully terminated. We need to enforce such a timeout to ensure that all student notebooks can be evaluated within a reasonable timeframe. In practice, this means that for your final submission you should *NOT* include the lengthy model training process. Instead, you'll have to find some way to train your model, save it into a persistent format and then load this trained model in your final submission *without* re-executing the training code. Every major machine learning library offers some pre-built functionality for this process and an explicit example implementation will be given further below.

✳️ **Limit Inference Times.** During the grading of your final submission, the predictions of your model will have to be evaluated on the full 750-sample evaluation set. Due to the aforementioned cell execution timeout (max. 5 minutes per cell), it is important that a single forward pass of your model doesn't take too long. Beware of this aspect when attempting to implement algorithmic solutions or extensive pre-processing steps.

✳️ **No Internet Access.** During the grading of your submission, there will only be restricted internet access. Specifically, the only reachable site will be the university's BwSync&Share Cloud. The cloud will be accessible to allow you to fetch trained versions of the model from your own file storage. More importantly, this means that you won't be able to use any external online services as part of your solution.

In [ ]:
##### DO NOT CHANGE #####
class DiffractogramInterface:
    """
    This class defines the Interface you'll need to implement / inherit
    from to solve the diffractogram regression task. Each subclass needs
    to override an implementation of the ``forward_image`` method, which
    takes a PIL.Image object as input and should return the predicted
    pair of material properties as a tuple ``(A, P)`` of two floats.
    """

    def forward_image(self, image: Image, **kwargs) -> Tuple[float, float]:
        """
        This function should be implemented in a subclass. It takes a
        ``PIL.Image`` object representing a (colored) diffractogram and
        returns the model's prediction of the two material properties
        as a tuple ``(A, P)`` of two floats:
          - ``A`` in [0, 10]   (amplitude)
          - ``P`` in [-3, 3]   (phase)
        """
        # 1. e.g. convert the Image object to a torch tensor
        # 2. run the forward pass of your model
        # 3. return the (A, P) prediction as a Python tuple of two floats

        raise NotImplementedError(
            "This method should be implemented in a subclass!"
        )


##### DO NOT CHANGE #####

## C.3 Loading the Data

The following section implements the data loading of the training dataset. The dataset is provided in the form of a ZIP file containing one PNG file and a JSON file for *each* sample. The PNG file holds the actual $512\times 512$ RGB diffractogram image; the JSON file holds the metadata for that sample — including the true target values $A$ and $P$.

The functions defined in the following section will implement most of the data loading. Ultimately the dataset will be returned in a dictionary structure, where the keys are the sample IDs and the values are dictionaries containing the image information and the metadata information in the following structure:
```python
index_data_map = {
    # ...
    0: {
        "image": <PIL.Image.Image>,  # 512x512 RGB diffractogram
        "data": {
            "index": 0,             # sample index
            "A":      6.83,         # true amplitude in [0, 10]
            "P":     -1.27,         # true phase     in [-3, 3]
        }
    },
}
```

In [ ]:
##### DO NOT CHANGE #####
def load_diffractogram_dataset(path: str) -> Dict[int, dict]:
    """
    Given the absolute string ``path`` to a folder containing the dataset, 
    this function will load the dataset and return a index data map dictionary 
    structure containing the images and the correspondign target value 
    annotations.
    
    The dataset folder is expected to contain a set of files where each element 
    of the dataset is represented by a pair of files: a PNG image and a JSON
    file. The PNG image contains the diffractogram, while the JSON file
    contains the target value for the corresponding image. The files are
    named using the same index, e.g.:
    - 0.png, 0.json
    
    :param path: The absolute path to the folder.
    
    :return: A dicionary mapping the index of the dataset element to a 
        dictionary containing the image and the target value.
    """
    
    # We'll use this data structure to store the dataset as we load 
    # it from the folder structure.
    index_data_map: Dict[int, dict] = {}
    
    # This set will contain all the string representations of the 
    # dataset indices.
    indices: Set[str] = set([
        name.split('.')[0]
        for name in os.listdir(path)
    ])
    
    for index in sorted(indices):
        json_path = os.path.join(path, f"{index}.json")
        image_path = os.path.join(path, f"{index}.png")
        
        # Load the image
        image = Image.open(image_path)

        # Load the json file
        with open(json_path, 'r') as f:
            data = json.load(f)
            
        index_data_map[int(index)] = {
            'image': image,
            'data': data
        }
        
    return index_data_map


def download_dataset(url: str, name: str) -> Dict[int, dict]:
    """
    Given the absolute string ``url`` to the dataset as a Nextcloud download 
    link, this function will download the dataset and return a index data map 
    dictionary structure containing the images and the correspondign target
    value annotations.
    
    :param url: The absolute URL to the dataset.
    :param name: The name of the dataset.
    
    :returns: A dicionary mapping the index of the dataset element to a
        dictionary containing the image and the target value.
    """
    
    with tempfile.TemporaryDirectory() as temp_dir:
        
        # Write the content of the temporary ZIP file into a file
        zip_path = os.path.join(temp_dir, f"{name}.zip")
        content = nextcloud_download(url, raw=True)
        with open(zip_path, 'wb') as file:
            file.write(content)
        
        # Unzip the file
        dataset_path = temp_dir
        print(f"> Unzipping {zip_path} to {dataset_path}")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(dataset_path)
            
        # Load the dataset
        dataset = load_diffractogram_dataset(os.path.join(dataset_path, name))

    return dataset

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# -- LOAD TRAIN SET --
# Here we load the training set in the format of a index-data map. The keys of 
# this data structure are the integer indices of the individual dataset elements and 
# the values are again dictionaries containing the input images and the associated 
# target value labels:
# - 'image': The input image as a PIL.Image object
# - 'data': A dictionary containing the sample metadata with the following keys:
#   - 'index': The integer index of the sample
#   - 'A': The true amplitude target value (float in [0, 10])
#   - 'P': The true phase target value (float in [-3, 3])

print('> loading the train set...')
index_data_map_train: Dict[int, dict] = download_dataset(
    url='https://bwsyncandshare.kit.edu/s/m3jQrwqMTWpFztt/download/train.zip',
    name='train',
)
print(f'✅ loaded dataset with {len(index_data_map_train)} entries.')

print('example data:')
example_data = index_data_map_train[0]
pprint(example_data)

##### DO NOT CHANGE #####

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task C.1.</strong> Explore the training dataset. Use the cell below to inspect a few example diffractograms and their target values, and to experiment with any pre-processing ideas that might later help your model cope with the distortions.
</div>

In [ ]:
# e.g Data exploration

# YOUR CODE HERE
raise NotImplementedError()

The following cell downloads the 25 elements of the **basement (easy)** subset that have been manually annotated by your collaborators and which you may use to validate your model. Make sure you understand which kinds of distortions are present in the dataset and how they may impact the performance of your approach.

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task C.2.</strong> Study the 25 annotated basement samples downloaded below. Try to identify the different types of distortion present — these are exactly the distortion types your model will be evaluated on. Understanding them is key to building a robust solution.
</div>

In [ ]:
##### DO NOT CHANGE #####
# Loading some samples from the basement set 

print('loading the basement validation samples...')
index_data_map_sample: Dict[int, dict] = download_dataset(
    url='https://bwsyncandshare.kit.edu/s/XBe7BC8KwBbbMZQ/download/test_sample.zip',
    name='test_sample',
)

images_sample: List[Image_] = [data['image'] for index, data in index_data_map_sample.items()]

# Plotting the sample images
fig, axes = plt.subplots(5, 5, figsize=(15, 15))
for idx, ax in enumerate(axes.flat):
    if idx < len(images_sample):
        ax.imshow(images_sample[idx])
        ax.axis('off')
    else:
        ax.axis('off')
plt.tight_layout()
plt.show()

##### DO NOT CHANGE #####

In [ ]:
# e.g. inspect the basement samples
# (apply your pre-processing pipeline to all 25 validation samples and look
# for the kinds of distortions that the held-out grading set will throw at
# your model)

# YOUR CODE HERE
raise NotImplementedError()


## C.4 Defining Your Model

### Mock Implementation

The following cell showcases a mock implementation of the ``DiffractogramInterface`` using a PyTorch Lightning ``LightningModule``. The mock implementation is a simple fully-connected network which takes the flattened image as input and produces the two output values (the predicted properties $A$ and $P$). This implementation is not expected to yield good results, but it serves to illustrate how to implement the interface.

Perhaps more specifically, the mock implementation also showcases how to *save* and *load* a trained model to and from a file format. This will be important for your final submission, as you'll have to save your trained model and load it from your nextcloud file storage during the grading of your submission! You may use the following implementations of the ``save`` and ``load`` methods as a template for your own model.

In [ ]:
##### DO NOT CHANGE #####

class MockDiffractogramModel(pl.LightningModule, DiffractogramInterface):
    """
    This mock implementation illustrates how to use the ``DiffractorgramInterface`` 
    to build a very simple pytorch lightning ``LightningModule`` model for the processing 
    of the diffractogram images.
    
    The model consists of a simple fully connected neural network with a single hidden layer.
    The forward pass of the model flattens the 2D image representation and passes it through 
    the network to obtain the two unconstrained output values (one for each property, A and P).
    """
    def __init__(self,
                 input_shape: Tuple[int, int],
                 output_size: int = 2,
                 ):
        
        pl.LightningModule.__init__(self)
        DiffractogramInterface.__init__(self)
        
        # ~ define hyperparameters
        
        self.input_shape = input_shape
        self.output_size = output_size
        
        self.hparams.update({
            'input_shape': input_shape,
            'output_size': output_size
        })
        
        # ~ define network architecture
        
        flat_size = np.prod(self.input_shape)
        self.lay_pred = nn.Sequential(
            nn.Linear(flat_size, 128),
            nn.ReLU(),
            nn.Linear(128, output_size),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        This function takes a batch of images in the format of a torch tensor 
        and returns the batched tensor prediction.
        
        :param x: The input image as a torch tensor.
        
        :returns: The model prediction as a batched torch tensor.
        """
        x = x.view(x.size(0), -1)
        out = self.lay_pred(x)
        return out
    
    def forward_image(self, image: Image, **kwargs) -> Tuple[float, float]:
        """Mock forward pass: ignore the input image and return the
        midpoints of the (A, P) target ranges. The internal MLP exists
        only to demonstrate the save / load pattern below."""
        # NOTE: replace this in your own model with an actual prediction
        # derived from ``image`` (e.g. via the ``self.forward`` MLP).
        return 5.0, 0.0
    
    # -- SAVING AND LOADING --
    # Pytorch Lightning provides a relatively convenient way to save and load a model 
    # to and from a checkpoint file. The checkpoint file will contain the model's weights
    # at the point of saving, as well as the hyperparameters (constructor arguments).
    # NOTE: To make sure that the saving and loading works, we need to make sure that 
    #       ``hparams`` attriubute (part of the ``LightningModule``) is updated with all 
    #       of the constructor arguments relevant to the construction of the model
    #       architecture.
    
    def save(self, path: str) -> None:
        """Saves the current version of the model to the given absolute file ``path``.

        We write a plain ``torch.save`` checkpoint containing just the state_dict
        and the two hyperparameters needed to reconstruct the architecture. This
        avoids the ``weights_only=True`` unpickling restriction that newer
        versions of ``torch.load`` impose on Lightning's ``AttributeDict``.
        """
        torch.save({
            'state_dict':   self.state_dict(),
            'input_shape':  self.input_shape,
            'output_size':  self.output_size,
        }, path)
    
    @classmethod
    def load(cls, path: str) -> 'MockDiffractogramModel':
        """Loads and returns a model instance from the given absolute file ``path``
        pointing to an existing checkpoint file (as written by ``save`` above)."""
        checkpoint = torch.load(path, weights_only=False)
        model = cls(
            input_shape=checkpoint['input_shape'],
            output_size=checkpoint['output_size'],
        )
        model.load_state_dict(checkpoint['state_dict'])
        return model

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ~ constructing the model
model = MockDiffractogramModel(
    input_shape=(512, 512, 3),
    output_size=2,
)

# ~ example usage
example = index_data_map_train[0]
output = model.forward_image(image=example['image'])
print(f'* example model output: {output}')

assert isinstance(output, tuple) and len(output) == 2, (
    "The output of forward_image must be a 2-tuple (A, P)!"
)
A_pred, P_pred = output
assert isinstance(A_pred, float), "A must be a Python float."
assert isinstance(P_pred, float), "P must be a Python float."

# ~ Saving and loading the model
model_path = os.path.join(PATH, 'model.ckpt')
model.save(model_path)

model_loaded = MockDiffractogramModel.load(model_path)
output_loaded = model_loaded.forward_image(image=example['image'])

assert output == output_loaded, (
    "The output of the loaded model should match the original."
)


##### DO NOT CHANGE #####

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task C.3.</strong> Define your own model that implements the <code>DiffractogramInterface</code> (i.e. provides a <code>forward_image</code> method returning the predicted $(A, P)$ pair). You may use the mock implementation above as a template, including its save/load logic.
</div>

In [ ]:
# e.g. define your model

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# e.g. instantiate your model class

# YOUR CODE HERE
raise NotImplementedError()

## C.5 Training & Saving Your Model

You may use the following section to define your model architecture and to implement a training loop.

<div style="border: 1px solid #C75949; border-radius: 3px; padding: 6px; background-color: #fbeae5; color: black;">
<strong>⚠️ NOTE — Remove Training Code.</strong> While you can use the following cells for the training of your model, keep in mind that you <em>should not remove or add any cells</em>. Additionally, make sure to <em>disable</em> your training code before submitting your solution (due to the cell execution timeout)!
</div>

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task C.4.</strong> Train your model on the training dataset and save the trained weights to a file. Upload the checkpoint to your BwSync&amp;Share storage so it can be downloaded during grading (see C.7). Remember to disable the training code before submitting.
</div>

In [ ]:
# e.g. implemenet the training loop

# YOUR CODE HERE
raise NotImplementedError()

## C.6 Validating Your Model

You may use this section to evaluate your model.

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task C.5.</strong> Evaluate your trained model. Inspect the training loss, check the performance on the (clean) training data, and — most importantly — measure the $R^2$ score on the 25 basement validation samples to estimate how well your model handles the distribution shift.
</div>

In [ ]:
# e.g. plot the training loss of the model

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# e.g. evaluate the model performance on train set

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# e.g. evaluate the model performance on the sample test elements

# YOUR CODE HERE
raise NotImplementedError()

## C.7 Final Submission

The following section defines your final submission to the competition. Make sure to assign an instance of your trained model to the ``MODEL`` variable below.

### Loading from BwSync&Share

Due to the cell execution timeout of 5 minutes, it won't be possible to train your model during the grading of your submission. Instead, you'll have to save your trained model to a file and load it from your nextcloud file storage during the grading. To do this, you'll first have to export your trained model to a file using a ``save`` method or similar functionality provided by the machine learning framework of your choice. Then you'll have to upload the checkpoint file to your nextcloud file storage at https://bwsyncandshare.kit.edu/.

After uploading the file, you'll have to create a *public sharing link* for the file. See the image below for an example of how to do this: First click the share symbol to open the share dialog, then click the "+" button next to the "Share Link" field. Pressing this button will automatically create the public URL and copy it to your clipboard.

<img src="https://bwsyncandshare.kit.edu/s/PiXDbKTgxe3nPBk/download" alt="Download Link">

<div style="border: 1px solid #C75949; border-radius: 3px; padding: 6px; background-color: #fbeae5; color: black;">
<strong>⚠️ IMPORTANT — your share link must be PUBLIC and PASSWORD-FREE.</strong>
For the grading to be able to download your model, the BwSync&Share link you paste below must be
configured correctly in the BwSync&Share share menu. Make sure you:
<ul>
<li>create a <strong>public "Share link"</strong> (via the "+" button) — <em>not</em> an internal share to a specific user;</li>
<li>do <strong>NOT</strong> set a password on the link — a password-protected link cannot be downloaded automatically during grading;</li>
<li>do <strong>NOT</strong> set an expiration date that falls before the grading is done — leave it empty or set it well past the submission deadline;</li>
<li>leave the link at read/download access and append the <code>/download</code> suffix to the URL (as shown below).</li>
</ul>
If the link is private, password-protected, expired or otherwise misconfigured, the grading cannot
load your model and you will receive <strong>0 points</strong> — even if your model is perfect.
</div>

After obtaining the public URL, you can use the following code snippet to download the file content and load the model checkpoint:
```python

with tempfile.TemporaryDirectory() as temp_path:

    file_path = os.path.join(temp_path, "model.ckpt")
    with open(file_path, "wb") as file:
        
        # NOTE: You'll have to replace with your own URL and not forget to add the "/download" suffix!
        content = nextcloud_download(
            url="{your_url}/download",
            raw=True
        )
        file.write(content)

        # Load the model checkpoint from the file...

```

### Pre-Submission Checklist 📋

- [ ] Does your notebook still contain the required nbgrader metadata?
- [ ] Are you certain that you have not added or removed any cells? 
- [ ] Have you removed/disabled the training code?
- [ ] Have you defined your model in the cell below?
- [ ] Are you loading your model in such a way that it will work in the grading environment? E.g. by downloading 
the checkpoint from nextcloud?
- [ ] Does your model implementation pass the public test cases below?

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task C.6.</strong> Load your trained model (e.g. by downloading the checkpoint from BwSync&amp;Share) and assign the ready-to-use instance to the <code>MODEL</code> variable in the cell below. This is the model that will be graded.
</div>

In [ ]:
# NOTE: Make sure to load and assign a *working and trained* version of the model to
#       this ``MODEL`` variable!
#       The model assigned to this variable will be considered as your final submission
#       to the challenge.
MODEL: DiffractogramInterface = None

# YOUR CODE HERE
raise NotImplementedError()

The following cell declares ungraded, *public tests* only. These public tests shall help you to properly check your model for the submission. If your submitted model does not pass the following test cases, the submission will most likely fail! To guarantee a smooth submission make sure that your model properly adheres to the ``DiffractogramInterface``.

In [ ]:
##### DO NOT CHANGE #####
# There has to be some kind of model assigned to the MODEL variable.
assert MODEL is not None, (
    "The MODEL variable has not been assigned a model yet!"
)

# The model instance has to be a subclass of the DiffractogramInterface class
# and therefore implement the ``forward_image`` method.
assert isinstance(MODEL, DiffractogramInterface), (
    "The submitted MODEL variable has to be a subclass of DiffractogramInterface!"
)
assert hasattr(MODEL, 'forward_image'), (
    "The submitted MODEL variable has to implement the forward_image method!"
)

# It has to be possible to run a forward pass of the model using a
# PIL.Image object representing an RGB image of shape (512, 512, 3).
import numpy as np
from PIL import Image
example_image = Image.fromarray(
    np.random.randint(0, 256, (512, 512, 3), dtype=np.uint8)
)
output = MODEL.forward_image(image=example_image)
print(f'* example model output: {output}')

# The output of forward_image has to be a 2-tuple (A, P) of Python floats.
assert isinstance(output, tuple), (
    "* The output of forward_image must be a tuple (A, P)!"
)
assert len(output) == 2, (
    "* The output tuple must have exactly two elements (A, P)!"
)
A_pred, P_pred = output
assert isinstance(A_pred, float), (
    "* The first element (A) must be a Python float!"
)
assert isinstance(P_pred, float), (
    "* The second element (P) must be a Python float!"
)

# Each forward pass should be faster than ~1 second on average. If the
# forward pass is much slower the full hidden evaluation may exceed the
# cell execution timeout.
import time
start = time.time()
for _ in range(5):
    _ = MODEL.forward_image(image=example_image)
elapsed = (time.time() - start) / 5
print(f'* average forward_image time: {elapsed*1000:.1f} ms')
assert elapsed < 1.0, (
    f"* A single forward_image call should take less than 1 second "
    f"on average (got {elapsed:.2f}s). The hidden test set has 750 "
    f"samples and slow forward passes may exceed the cell timeout."
)


##### DO NOT CHANGE #####

## C.8 Grading

The following section contains the hidden tests related to the *bonus* grading part of the competition. The first three cells evaluate your model on each of the three held-out subsets and store the per-subset results. The remaining six cells check each subset's mean $R^2$ against its two ceilings (the lower and higher thresholds listed in C.1) and award the corresponding bonus points. Make sure not to modify any of these cells.

In [ ]:
##### DO NOT CHANGE #####
# ID: comp-eval-test - possible points: 0

# This cell runs your trained model on the hidden 'test' subset and measures
# how accurate its predictions are (the mean R² score). It does not award any
# points on its own — the two cells below use this result to decide your bonus points.



##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: comp-eval-easy - possible points: 0

# This cell runs your trained model on the hidden 'basement_easy' subset and measures
# how accurate its predictions are (the mean R² score). It does not award any
# points on its own — the two cells below use this result to decide your bonus points.



##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: comp-eval-hard - possible points: 0

# This cell runs your trained model on the hidden 'basement_hard' subset and measures
# how accurate its predictions are (the mean R² score). It does not award any
# points on its own — the two cells below use this result to decide your bonus points.



##### DO NOT CHANGE #####

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>Test set (clean).</strong> The two cells below evaluate your model on the 250-image <code>test</code> subset and award points based on its mean $\bar R^2 = \tfrac{1}{2}(R^2_A + R^2_P)$:
<ul>
<li><strong>Lower ceiling:</strong> $\bar R^2 \ge 0.80$ → <strong>5 pts</strong></li>
<li><strong>Higher ceiling:</strong> $\bar R^2 \ge 0.90$ → <strong>10 pts</strong></li>
</ul>
</div>

In [ ]:
##### DO NOT CHANGE #####
# ID: comp-test-low - possible points: 5

# This cell awards 5 points if your model's accuracy on the 'test' subset
# (the mean R² measured in the cell above) reaches the lower target of 0.8.



##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: comp-test-high - possible points: 10

# This cell awards 10 points if your model's accuracy on the 'test' subset
# (the mean R² measured in the cell above) reaches the higher target of 0.9.



##### DO NOT CHANGE #####

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>Basement (easy) — one distortion.</strong> The two cells below evaluate your model on the 250-image <code>basement_easy</code> subset and award points based on its mean $\bar R^2 = \tfrac{1}{2}(R^2_A + R^2_P)$:
<ul>
<li><strong>Lower ceiling:</strong> $\bar R^2 \ge 0.85$ → <strong>10 pts</strong></li>
<li><strong>Higher ceiling:</strong> $\bar R^2 \ge 0.95$ → <strong>15 pts</strong></li>
</ul>
</div>

In [ ]:
##### DO NOT CHANGE #####
# ID: comp-easy-low - possible points: 10

# This cell awards 10 points if your model's accuracy on the 'basement_easy' subset
# (the mean R² measured in the cell above) reaches the lower target of 0.85.



##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: comp-easy-high - possible points: 15

# This cell awards 15 points if your model's accuracy on the 'basement_easy' subset
# (the mean R² measured in the cell above) reaches the higher target of 0.95.



##### DO NOT CHANGE #####

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>Basement (hard) — two distortions.</strong> The two cells below evaluate your model on the 250-image <code>basement_hard</code> subset and award points based on its mean $\bar R^2 = \tfrac{1}{2}(R^2_A + R^2_P)$:
<ul>
<li><strong>Lower ceiling:</strong> $\bar R^2 \ge 0.75$ → <strong>10 pts</strong></li>
<li><strong>Higher ceiling:</strong> $\bar R^2 \ge 0.85$ → <strong>20 pts</strong></li>
</ul>
</div>

In [ ]:
##### DO NOT CHANGE #####
# ID: comp-hard-low - possible points: 10

# This cell awards 10 points if your model's accuracy on the 'basement_hard' subset
# (the mean R² measured in the cell above) reaches the lower target of 0.75.



##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: comp-hard-high - possible points: 20

# This cell awards 20 points if your model's accuracy on the 'basement_hard' subset
# (the mean R² measured in the cell above) reaches the higher target of 0.85.



##### DO NOT CHANGE #####

## C.9 Competition Ranking

The following section contains the hidden test related to the competition ranking. Make sure not to modify the cells!

In [ ]:
##### DO NOT CHANGE #####
# ID: comp-ranking - possible points: 0

# This cell saves your final scores and the total number of bonus points you earned.
# It also records the per-property accuracies that are used for the prize ranking.



##### DO NOT CHANGE #####